In [ ]:
# cell: nb_seg
# Kernel: Select Kernel → Python Environments → ~/venvs/venv_seg (dựng ở P2 mục 4.4).
import sys
from pathlib import Path
REPO = Path.home() / 'Documents' / 'nckh'       # sửa nếu bạn clone repo ở chỗ khác
ROOT = Path.home() / 'nckh_root'                # NCKH_ROOT (default của nckh.paths)
DATA = Path.home() / 'nckh_data'                # NCKH_LOCAL_DATA
PANDERM = Path.home() / 'PanDerm'
CK = ROOT / 'checkpoints' / 'panderm_bb_data6_checkpoint-499.pth'
PY = sys.executable                             # chính là venv_seg, dùng trong lệnh "!"
%cd {REPO}
!git log --oneline -1

In [ ]:
# cell: nb_seg — ghi SHA-256 checkpoint (tải ở P2 mục 4.3)
from datetime import date
from nckh.runcard import sha256_file
digest = sha256_file(CK)
print(digest)
with open(ROOT / 'checkpoints' / 'SHA256SUMS', 'a') as fh:
    fh.write(f'{digest} {CK.name} #tai {date.today()}\n')

In [ ]:
# cell: nb_seg
!{PY} scripts/inspect_checkpoint.py {CK}

In [ ]:
# cell: nb_seg
PATCH = REPO / 'patches' / 'panderm_base_seg.patch'
# Kiểm tra theo chiều ngược trước, để chạy lại cell này nhiều lần cũng không áp patch hai lần.
!cd {PANDERM} && (git apply --reverse --check {PATCH} 2>/dev/null && echo "Patch đã áp từ trước") || (git apply {PATCH} && echo "Đã áp patch")
!cd {PANDERM} && git diff --stat

In [ ]:
# cell: nb_seg — smoke test 20 ảnh rồi xem overlay
from IPython.display import Image, display
from nckh.runcard import new_run_id
RUN = ROOT / 'runs' / new_run_id('smoke_seg')
!{PY} scripts/bench_inference.py --task seg --panderm-dir {PANDERM}/segmentation --pretrained {CK} --images "{DATA}/smoke/ISIC2018_Task1-2_Validation_Input/*.jpg" --n 20 --save-overlays 10 --out-dir {RUN}
!{PY} -m nckh.runcard {RUN} --seed 0 --input pretrained={CK} --input patch={PATCH} --config task=smoke_seg --config n=20
for p in sorted((RUN / 'overlays').glob('*.png'))[:4]:
    display(Image(filename=str(p), width=320))